In [1]:
import helpers.data as data
import helpers.features as features
import joblib
import hmmlearn.hmm as hmm
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from main import gbt_features, hmm_features, train_size, sequence_length

In [2]:
df = data.get_data()
df = features.add_features(df)
df = df.dropna()

Features computed in 1.739s


In [3]:
# for f in hmm_features:
#     print(f"{f}  min: {df[f].min():.6g}  max: {df[f].max():.6g}  inf: {np.isinf(df[f]).sum()}")

# # Replace inf/-inf with NaN then drop — inf rows would crash the HMM fit
# hmm_train = df[hmm_features].iloc[:int(len(df) * train_size)].replace([np.inf, -np.inf], np.nan).dropna()
# print(f"\nHMM training rows: {len(hmm_train)}")

# hmm_model = hmm.GaussianHMM(n_components=3, covariance_type="diag")
# hmm_model.fit(hmm_train)
# joblib.dump(hmm_model, "./trained_models/regime_hmm.pkl")

In [4]:
hmm = joblib.load("./trained_models/regime_hmm.pkl")
df["hmm_state"] = hmm.predict(df[hmm_features])

In [5]:
# ── Regime-Weighted Sample Training ─────────────────────────────────────────────
# Compute per-bar sample weights that amplify training signal from bars matching
# deployment conditions: mean-reverting regime, extreme stretch, genuine vol normalcy.

# Get full training dataframe with all features needed for weight computation
train_df_full = df.iloc[:int(len(df) * train_size)].copy()

# 1. HMM state weight — identify mean-reverting state by lowest mean Hurst
#    Map states → {most_mr: 2.0, transitional: 0.8, trending: 0.25}
hmm_states = train_df_full["hmm_state"].values
hurst_vals = train_df_full["hurst"].fillna(0.5).values

# Compute mean Hurst per state to identify which is most mean-reverting
state_hurst_means = {}
for state in np.unique(hmm_states):
    mask = hmm_states == state
    if mask.sum() > 0:
        state_hurst_means[state] = np.mean(hurst_vals[mask])

# Find state with lowest mean Hurst (most mean-reverting)
most_mr_state = min(state_hurst_means, key=state_hurst_means.get)
state_weights_map = {most_mr_state: 2.0}
for state in state_hurst_means:
    if state != most_mr_state:
        # Classify as transitional (medium Hurst) or trending (high Hurst)
        if state_hurst_means[state] < 0.55:
            state_weights_map[state] = 0.8  # transitional
        else:
            state_weights_map[state] = 0.25  # trending

hmm_weight = np.array([state_weights_map.get(s, 1.0) for s in hmm_states])

# 2. Hurst weight — exponential penalty for trending bars (H > 0.5)
hurst_weight = np.exp(-5.0 * np.maximum(0, hurst_vals - 0.45))

# 3. EMA distance magnitude weight — extreme-stretch bars are more informative
ema_dist_abs_vals = train_df_full["ema_dist_abs"].fillna(0).values
ema_dist_mean = np.mean(ema_dist_abs_vals[ema_dist_abs_vals > 0])
if ema_dist_mean > 0:
    ema_dist_weight = np.clip(ema_dist_abs_vals / ema_dist_mean, 0.5, 4.0)
else:
    ema_dist_weight = np.ones_like(ema_dist_abs_vals)

# 4. EMA divergence change quality — upweight bars where reversion already started
ema_div_chg3_vals = train_df_full["ema_div_chg_3"].fillna(0).values
ema_div_chg_weight = np.where(ema_div_chg3_vals < 0, 1.5, 0.7)  # contracting = higher weight

# Combine weights multiplicatively, then normalize to mean=1
regime_weights = hmm_weight * hurst_weight * ema_dist_weight * ema_div_chg_weight
regime_weights = regime_weights / np.mean(regime_weights)

print(f"Regime weights computed:")
print(f"  Mean weight: {np.mean(regime_weights):.4f}")
print(f"  Min weight:  {np.min(regime_weights):.4f}")
print(f"  Max weight:  {np.max(regime_weights):.4f}")
print(f"  HMM state mapping: {state_weights_map}")
print(f"  Most mean-reverting state: {most_mr_state} (mean Hurst: {state_hurst_means[most_mr_state]:.4f})")

Regime weights computed:
  Mean weight: 1.0000
  Min weight:  0.0040
  Max weight:  9.2665
  HMM state mapping: {np.int64(1): 2.0, np.int64(0): 0.25, np.int64(2): 0.25}
  Most mean-reverting state: 1 (mean Hurst: 0.7834)


In [6]:
df = features.add_target(df)
train_df = df[gbt_features].iloc[:int(len(df) * train_size)]
targets = df['target'].iloc[:int(len(df) * train_size)].values

def create_sequences(data, target_values):
    feature_data = data[gbt_features].values
    X_sequences = []
    y_targets = []
    
    for i in range(sequence_length, len(data)):
        sequence = feature_data[i - sequence_length:i]
        flattened_sequence = sequence.flatten()
        X_sequences.append(flattened_sequence)
        y_targets.append(target_values[i])
    
    return np.array(X_sequences), np.array(y_targets)

X, Y = create_sequences(train_df, targets)
print(f"Training data shape: X={X.shape}, Y={Y.shape}")
print(f"Target stats: mean={Y.mean():.6f}, std={Y.std():.6f}, min={Y.min():.6f}, max={Y.max():.6f}")

# Align regime weights to sequence indices
# weights[i] = weight of bar i (the bar being predicted)
# X sequences start at index sequence_length, so slice weights accordingly
train_weights = regime_weights[sequence_length : sequence_length + len(X)]
print(f"Sample weights shape: {train_weights.shape}")
print(f"Sample weights stats: mean={train_weights.mean():.4f}, min={train_weights.min():.4f}, max={train_weights.max():.4f}")

model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    subsample=0.7,
    max_features='sqrt',
    loss='huber',  # Robust to outliers; targets are continuous in [-1, 1]
    validation_fraction=0.2,
    n_iter_no_change=25,
    random_state=39,
    verbose=1
)

model.fit(X, Y, sample_weight=train_weights)

# Get continuous predictions in [-1, 1]
train_pred = model.predict(X)

# Create test set from remaining data
test_df_full = df.iloc[int(len(df) * train_size):].copy()
test_df_full = test_df_full.dropna(subset=gbt_features + ['target'])
test_df = test_df_full[gbt_features]
test_targets = test_df_full['target'].values

X_test, Y_test = create_sequences(test_df, test_targets)

# Get continuous predictions for test set
test_pred = model.predict(X_test)

# Regression metrics
train_mae = mean_absolute_error(Y, train_pred)
test_mae = mean_absolute_error(Y_test, test_pred)
train_mse = mean_squared_error(Y, train_pred)
test_mse = mean_squared_error(Y_test, test_pred)
train_r2 = r2_score(Y, train_pred)
test_r2 = r2_score(Y_test, test_pred)

print(f"\nTraining Metrics:")
print(f"  MAE:  {train_mae:.6f}")
print(f"  RMSE: {np.sqrt(train_mse):.6f}")
print(f"  R²:   {train_r2:.6f}")

print(f"\nTest Metrics:")
print(f"  MAE:  {test_mae:.6f}")
print(f"  RMSE: {np.sqrt(test_mse):.6f}")
print(f"  R²:   {test_r2:.6f}")

print(f"\nPredictions - Train: [{train_pred.min():.6f}, {train_pred.max():.6f}]")
print(f"Predictions - Test:  [{test_pred.min():.6f}, {test_pred.max():.6f}]")
print(f"Target range - Train: [{Y.min():.6f}, {Y.max():.6f}]")
print(f"Target range - Test:  [{Y_test.min():.6f}, {Y_test.max():.6f}]")

# Feature importance analysis
feature_importance = model.feature_importances_
feature_names_expanded = [f"{feat}_t-{i}" for i in range(sequence_length-1, -1, -1) for feat in gbt_features]
top_indices = np.argsort(feature_importance)[-20:][::-1]
print(f"\nTop 20 Most Important Features:")
for idx in top_indices:
    print(f"  {feature_names_expanded[idx]}: {feature_importance[idx]:.6f}")

# Aggregate importance by original feature
feature_importance_agg = np.zeros(len(gbt_features))
for i, feat in enumerate(gbt_features):
    start_idx = i * sequence_length
    end_idx = (i + 1) * sequence_length
    feature_importance_agg[i] = feature_importance[start_idx:end_idx].sum()

print(f"\nAggregated Feature Importance:")
sorted_feat_idx = np.argsort(feature_importance_agg)[::-1]
for idx in sorted_feat_idx:
    print(f"  {gbt_features[idx]}: {feature_importance_agg[idx]:.6f}")

joblib.dump(model, "./trained_models/gbt.pkl")
print(f"\nModel saved. Best iteration: {model.n_estimators_}")

Training data shape: X=(40992, 208), Y=(40992,)
Target stats: mean=-0.000228, std=0.907236, min=-5.000000, max=5.000000
Sample weights shape: (40992,)
Sample weights stats: mean=1.0003, min=0.0040, max=9.2665
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           0.2403          -0.0656           27.70s
         2           0.2258          -0.0135           25.74s
         3           0.2246           0.0114           24.20s
         4           0.2209          -0.0010           24.06s
         5           0.2159           0.0037           24.06s
         6           0.2094          -0.0022           23.97s
         7           0.2117           0.0165           23.82s
         8           0.2087           0.0047           23.80s
         9           0.2079           0.0075           23.83s
        10           0.1966          -0.0158           23.74s
        20           0.1769          -0.0145           22.10s
        30           0.1669          -0.0068  